# Add hexagonal grid coordinates to GBIF

Set path to include RStudio packages

In [2]:
.libPaths(c("/net/home/swolf/R/x86_64-pc-linux-gnu-library/4.4", .libPaths()))

In [3]:
library(data.table)
dtn = 10
setDTthreads(threads = dtn) #defines how many threads data.table is allowed to use

In [ ]:
# Citizen science data from GBIF, cleaned and filtered to species with at least one trait value in TRY6 gap-filled dataset
gbif_dt_filtered <- fread("/net/scratch/swolf/GBIF/gbif_try_filtered_cs.csv")
head(gbif_dt_filtered)

gbifID,speciesKey,decimalLatitude,decimalLongitude
<int64>,<int>,<dbl>,<dbl>
3352641301,3103791,56.14011,-3.369429
4951294464,8059811,54.97456,-1.634406
4866764380,2878688,54.99222,-1.540470
3098399480,5361779,52.75213,1.511204
5158856302,8377689,51.60241,-1.122803
3098399483,5371825,51.10395,-1.993522


In [ ]:
# Add hexagonal grid coordinates to GBIF data using the h3 package

library(data.table)
library(h3)

# Set parameters
resolution_3 <- 3  # H3 resolution level 3
resolution_4 <- 4  # H3 resolution level 4
chunk_size <- 1e6  # Process data in 1 million row chunks

# Output file path
#output_file <- "/net/scratch/swolf/GBIF/gbif_h3/res3_res4.csv"
output_file <- "/net/scratch/swolf/GBIF/gbif_h3/res3_res4_cs.csv"

# Ensure the dataset is a data.table
setDT(gbif_dt_filtered)

# Remove existing output file to prevent appending issues
if (file.exists(output_file)) {
    file.remove(output_file)
}

# Get total rows
total_rows <- nrow(gbif_dt_filtered)

# Process data in chunks
for (start_row in seq(1, total_rows, by = chunk_size)) {
    # Determine end row for the current chunk
    end_row <- min(start_row + chunk_size - 1, total_rows)
    
    # Extract the chunk
    chunk <- gbif_dt_filtered[start_row:end_row]
    
    # Apply geo_to_h3 function and store results for both resolutions
    chunk[, h3_res3 := geo_to_h3(c(decimalLatitude, decimalLongitude), resolution_3)]
    chunk[, h3_res4 := geo_to_h3(c(decimalLatitude, decimalLongitude), resolution_4)]
    
    # Save the processed chunk (append mode)
    fwrite(chunk, output_file, append = TRUE)
    
    # Print progress
    cat("Processed rows:", start_row, "to", end_row, "\n")
}

# Optional: Load the processed results back into a data.table if needed
# processed_data <- fread(output_file)


# Merge with trait data from TRY

In [1]:
#gbif_h3 <- fread("/net/scratch/swolf/GBIF/gbif_h3/res3_res4.csv", na.strings = "", quote = "")
gbif_h3 <- fread("/net/scratch/swolf/GBIF/gbif_h3/res3_res4_cs.csv", na.strings = "", quote = "")

head(gbif_h3)

ERROR: Error in fread("/net/scratch/swolf/GBIF/gbif_h3/res3_res4_cs.csv", na.strings = "", : could not find function "fread"


In [6]:
# TRY6
TRY_species_means <- fread("../Results/Data/01_try6_gapfilled_species_means.csv")
head(TRY_species_means)

speciesKey,X.X3117.,X.X14.,X.X50.,X.X146.,X.X3120.,X.X224.,X.X297.,X.X21.,X.X13.,⋯,X.X223.,X.X237.,X.X614.,X.X351.,X.X6.,X.X95.,X.X138.,X.X1080.,X.X78.,X.X4.
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2393,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6655,9.321433,NA,2.149281,NA,2.784474,NA,NA,0.007640780,NA,⋯,32.96780,8.897962,NA,0.7183379,NA,NA,541.0121,2647.8870,NA,0.7780896
6657,17.527908,25.95187,1.423995,18.99770,2.455510,NA,13.749035,0.004666096,NA,⋯,31.58875,NA,2390.416,NA,NA,NA,2327.8265,3590.6496,NA,0.6077446
6752,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
7717,12.245032,21.21620,2.022381,22.12193,NA,3.672329,14.885849,0.040939324,NA,⋯,NA,4.331903,NA,1.5688087,1.166276,93.53081,NA,6500.2818,4.228426,0.5170459
1010512,9.473567,17.16218,2.126191,24.66683,2.384060,1.338781,7.013938,0.134267482,414.5069,⋯,27.28636,1.665713,2630.476,1.0977119,1.895727,87.15040,189.7187,637.9941,1.564110,0.4896163


In [7]:
# Join on speciesKey
gbif_try <- merge(gbif_h3, TRY_species_means, by = "speciesKey")

head(gbif_try)

speciesKey,gbifID,decimalLatitude,decimalLongitude,h3_res3,h3_res4,X.X3117.,X.X14.,X.X50.,X.X146.,⋯,X.X223.,X.X237.,X.X614.,X.X351.,X.X6.,X.X95.,X.X138.,X.X1080.,X.X78.,X.X4.
<int>,<int64>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1204404,5167350772,-5.840646,12.252583,838230fffffffff,8482301ffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA
1204404,4987455975,-6.185731,12.318853,838230fffffffff,8482309ffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA
1204404,2445057444,-9.331337,13.159449,83831efffffffff,84831ebffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA
1204404,2851019390,-9.195712,13.274815,83831efffffffff,84831ebffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA
1204404,2465115388,-2.414163,9.638134,838218fffffffff,8482189ffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA
1204404,2988502417,-4.977251,11.976894,838233fffffffff,848233dffffffff,5.765383,13.15323,2.449195,36.6421,⋯,NA,25.00274,NA,1.011806,NA,86.26981,129.1691,NA,NA,NA


Check that all observations were matched with TRY data, which is to be expected, because we prefiltered GBIF for TRY species in preprocessing --\> did this with TRY6 species list!


In [8]:
dim(gbif_h3) # cs only
dim(gbif_try) # cs only

[1] 64141358        6

[1] 64141358       43

In [9]:
fwrite(gbif_try, "/net/scratch/swolf/GBIF/gbif_res4_res3_try6_cs.csv", row.names = F, quote = T)